# Lumpy 01: Data Check

Notebook version: 3

Loads the lumpy demand file, checks the external feature handoff, builds the model frame, and creates the rolling split table.


In [ ]:
import time
KERNEL_START_TIME = time.perf_counter()
print("OK: kernel is running")


In [ ]:
from pathlib import Path
import importlib
import sys
import time

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    candidates = [
        start,
        start.parent,
        start.parent.parent,
        start / "Inchscape Repo",
        start.parent / "Inchscape Repo",
    ]
    for candidate in candidates:
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not find repo root. Open this notebook from the Inchscape Repo folder "
        "or update find_repo_root() with the project path."
    )


root = find_repo_root()
src_path = str(root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

_lumpy = importlib.import_module("lumpy_forecasting")

LumpyConfig = _lumpy.LumpyConfig
build_lumpy_model_frame = _lumpy.build_lumpy_model_frame
load_lumpy_inputs = _lumpy.load_lumpy_inputs
make_backtest_splits = _lumpy.make_backtest_splits

print("OK: lumpy_forecasting imported")
print(f"Project root: {root}")
print(f"Module path: {Path(_lumpy.__file__).resolve()}")


In [ ]:
# Keep this as a read-only check by default. The build notebook writes final outputs.
SAVE_DATA_CHECK_OUTPUTS = False

start_time = time.perf_counter()
print("Loading lumpy inputs and building the data-check model frame...")

config = LumpyConfig(max_folds=None, external_mode="selected")
sales, external = load_lumpy_inputs(root, config)
model_data, external_inventory = build_lumpy_model_frame(sales, external, config)
splits = make_backtest_splits(model_data, config)

elapsed = time.perf_counter() - start_time
print(f"OK: data check ready in {elapsed:.1f}s")
print("Sales rows:", len(sales))
print("Model rows:", len(model_data))
print("SKUs:", model_data["sku_id"].nunique())
print("Month range:", model_data["month"].min(), "to", model_data["month"].max())
print("External features:", len(external_inventory))
print("Backtest splits:", len(splits))

display(
    model_data
    .groupby("month", as_index=False)
    .agg(total_demand=("demand", "sum"), active_skus=("sku_id", "nunique"))
    .tail()
)
display(external_inventory)
display(splits)


In [ ]:
# Optional: write the prepared data and split table from this check notebook.
# Usually leave this off and let the build notebook write final outputs.
if SAVE_DATA_CHECK_OUTPUTS:
    start_time = time.perf_counter()

    prepared_paths = {
        "model_data": root / "results" / "lumpy_outputs" / "tables" / "lumpy_model_data.csv",
        "splits": root / "results" / "lumpy_outputs" / "tables" / "lumpy_backtest_splits.csv",
        "external_inventory": root / "results" / "lumpy_outputs" / "tables" / "lumpy_selected_external_features.csv",
    }
    prepared_paths["model_data"].parent.mkdir(parents=True, exist_ok=True)
    model_data.to_csv(prepared_paths["model_data"], index=False)
    splits.to_csv(prepared_paths["splits"], index=False)
    external_inventory.to_csv(prepared_paths["external_inventory"], index=False)

    elapsed = time.perf_counter() - start_time
    print(f"OK: wrote data-check outputs in {elapsed:.1f}s")
    display(prepared_paths)
else:
    print("SAVE_DATA_CHECK_OUTPUTS is False, so this notebook did not overwrite output files.")
    print("Use the build notebook to write final lumpy outputs.")
